**import's**

In [1]:
!pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import random, warnings
from collections import Counter
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from xgboost import XGBClassifier

#Try to import PyTorch for LSTM
import torch
import torch.nn as nn
from  torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Pytorch {torch.__version__} available - LSTM models enabled (device: {device})")

Pytorch 2.10.0+cu128 available - LSTM models enabled (device: cuda)


**config**

In [3]:
NUM_RANGE = 49
N_SPLITS = 5
RANDOM_STATE = 42

**load  data**

In [4]:
def load_data(path):
  df = pd.read_csv(path, parse_dates=['Date'])
  df = df.sort_values('Date', ascending=False).reset_index(drop=True)
  return df

from google.colab import drive
drive.mount('/content/drive')

df = load_data('/content/drive/MyDrive/Colab Notebooks/sg_pools/ToTo.csv')
display(df.head(2))

Mounted at /content/drive


,Draw,Date,Winning Number 1,2,3,4,5,6,Additional Number,From Last,...,Division 3 Winners,Division 3 Prize,Division 4 Winners,Division 4 Prize,Division 5 Winners,Division 5 Prize,Division 6 Winners,Division 6 Prize,Division 7 Winners,Division 7 Prize
0,4178,2026-04-30,2,6,7,31,35,39,15,NaN,...,223.0,1818.0,639.0,346.0,12572.0,50.0,18347.0,25.0,238219.0,10.0
1,4177,2026-04-27,3,11,13,22,28,48,21,3,...,163.0,1522.0,420.0,323.0,8770.0,50.0,11113.0,25.0,159071.0,10.0


**transform**

In [5]:
def create_binary_matrix(df):
  records = []

  for _, row in df.iterrows():
    # All 7 winning numbers  (6 main + additional number treated as 7th)
    # the unified approach treats additional number  the  same as main numbers
    nums = set(row[['Winning Number 1','2','3','4','5','6']])
    additional_num = row['Additional Number']
    all_nums =  nums | {additional_num}

    # create binary columns for  all numbers
    record = {f"num_{i}": (1 if i in all_nums else 0) for i in range(1, NUM_RANGE+1)}

    record['Date'] = row['Date']
    records.append(record)
  return pd.DataFrame( records)

create_binary_matrix(df.head(2))

,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,num_10,...,num_41,num_42,num_43,num_44,num_45,num_46,num_47,num_48,num_49,Date
0,0,1,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-04-30
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,2026-04-27


**feature engineering**

In [6]:
def add_features(bin_df):
  df = bin_df.copy()
  feature_dict = {} # create all feature columns to odd at ocne (better performance)

  for i in range(1, NUM_RANGE+1):
    col = f"num_{i}"

    # Rolling frequencies - SHIFTED by 1 to avoid data leakage
    # We use shift(1) so that at time t, we onlyuse data from t-1 and before
    feature_dict[f"{col}_freq_10"] = df[col].shift(1).rolling(10, min_periods=1).mean()
    feature_dict[f"{col}_freq_30"] = df[col].shift(1).rolling(30, min_periods=1).mean()
    feature_dict[f"{col}_freq_50"] = df[col].shift(1).rolling(50, min_periods=1).mean()

    # lag features (already shifted)
    feature_dict[f"{col}_lag1"] = df[col].shift(1)
    feature_dict[f"{col}_lag2"] = df[col].shift(2)
    feature_dict[f"{col}_lag3"] = df[col].shift(3)

    # gap since last appeared
    gap, last_seen = [], -1
    for idx in range(len(df)):
      # Look at previous rows only (shifted by 1)
      if idx > 0 and df[col].iloc[idx-1] == 1: last_seen = idx - 1
      gap.append(idx - last_seen if last_seen != -1 else idx)

    feature_dict[f"{col}_gap"] = gap
    feature_dict[f"{col}_ema_10"] = df[col].shift(1).ewm(span=10, min_periods=1).mean()

  # Advanced feature: Number co-occurence patterns
  # Track which number tend to appear together
  for i in range(1, NUM_RANGE+1):
    col_i = f"num_{i}"
    # Co-occurence with nearby numbers (within 3)
    nearby_cols = [f"num_{j}" for j in range(max(1, i-5), min(NUM_RANGE+1, i+6)) if j != i]
    if nearby_cols:
      nearby_sum = df[nearby_cols].sum(axis=1).shift(1)
      feature_dict[f"{col_i}_nearby_cooccur"] = nearby_sum.rolling(20, min_periods=1).mean()

    # Global features (same for all members but useful context)
    # sum of all numbers in previous draws
    all_num_cols = [f"num_{i}" for i in range(1, NUM_RANGE+1)]
    total_sum = df[all_num_cols].sum(axis=1)
    feature_dict['prev_draw_sum'] = total_sum.shift(1)
    feature_dict['prev_draw_sum_ma5'] = total_sum.shift(1).rolling(5, min_periods=1).mean()

    # Odd/Even ratio in previous draws
    odd_cols = [f"num_{i}" for i in range(1, NUM_RANGE+1, 2)]
    even_cols = [f"num_{i}" for i in range(2, NUM_RANGE+1, 2)]
    feature_dict['prev_odd_count'] = df[odd_cols].sum(axis=1).shift(1)
    feature_dict['prev_even_count'] = df[even_cols].sum(axis=1).shift(1)

    # Low/High ratio (1-25 vs 26-49)
    low_cols = [f"num_{i}" for i in range(1, 26)]
    high_cols = [f"num_{i}" for i in range(26, NUM_RANGE+1)]
    feature_dict['prev_low_count'] = df[low_cols].sum(axis=1).shift(1)
    feature_dict['prev_high_count'] = df[high_cols].sum(axis=1).shift(1)

    # Decode distribution (1-10, 11-20, etc.)
    for decode in range(5):
      start = decode * 10 + 1
      end = min((decode + 1) * 10, NUM_RANGE)
      decode_cols = [f"num_{i}" for i in range(start, end+1)]
      feature_dict[f"prev_decode_{decode}_count"] = df[decode_cols].sum(axis=1).shift(1)

    # Streak features - how many draws since a number appeared
    for i in range(1, NUM_RANGE+1):
      col = f"num_{i}"
      # Calculate cumulative count of draws since last appearnce
      appeared = df[col].shift(1)
      feature_dict[f"{col}_streak"] = appeared.groupby((appeared == 1).cumsum()).cumcount()

    # add all features at once using pd.concat
    feature_df = pd.DataFrame(feature_dict)
    df = pd.concat([df, feature_df], axis=1)

  # Drop rows with NaN (first few rows)
  df = df.dropna().reset_index(drop=True)
  return df

add_features(create_binary_matrix(df.head()))

,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,num_10,...,num_40_nearby_cooccur,num_41_nearby_cooccur,num_42_nearby_cooccur,num_43_nearby_cooccur,num_44_nearby_cooccur,num_45_nearby_cooccur,num_46_nearby_cooccur,num_47_nearby_cooccur,num_48_nearby_cooccur,num_49_nearby_cooccur
0,0,0,1,0,1,0,0,1,1,0,...,1.333333,1.0,0.666667,1.333333,1.333333,1.00,1.00,1.00,0.0,0.666667
1,1,0,1,0,0,1,0,0,0,0,...,1.250000,1.0,0.750000,1.000000,1.500000,1.25,1.25,1.25,0.5,0.500000


In [7]:
class LotteryLSTM(nn.Module):
  """
    LSTM  model for lottery  number  prediction.


    Architecture:
      -  Input: (batch_size, seq_len, input_size)
      -  2-layer LSTM with dropout
      -  Fully connected layer with output size
      -  Sigmoid activation for binary classification
  """
  def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
    super(LSTMModel, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.lstm =  nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0
    )
    self.dropout = nn.Dropout(dropout)
    self.fc1 = nn.Linear(hidden_size, 64)
    self.fc2 = nn.Linear(64, 1)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLU()

  def forward(self, x):
    lstm_out, (h_n, c_n) = self.lstm(x)
    lstm_hidden  = lstm_out[:,-1,:]
    out  = self.dropout(lstm_hidden)
    out  = self.relu(self.fc1(out))
    out  = self.dropout(out)
    out  = self.sigmoid(self.fc2(out))
    return out

def create_sequences(data, targets, seq_length=20):
  """
    Create sequence for LSTM training

    Args:
      data: feature matrix (n_samples, n_features)
      targets: target vector (n_samples, )
      seq_length: Number of time steps to look back for each

    Returns:
      x: Sequences (n_sequences, seq_length, n_features)
      y: Targets (n_sequences, )

  """
  X, y = [], []
  for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    y.append(targets[i+seq_length])
  return np.array(X), np.array(y)

def train_lstm_model(X_train, y_train, input_size, epochs=50, batch_size=32, lr=0.001):
  """
    Train LSTM model

    Args:
      X_train: Training Sequences
      y_train: Training Targets
      input_size: Number of features
      epochs: Training epochs
      batch_size: Batch Size
      lr: learining rate

    Returns:
      Trained LSTM model
  """

  model = LotteryLSTM(input_size=input_size).to(device)
  criterion = nn.BCELoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=lr)

  # Convert to Tensors
  X_train_tensor = torch.tensor(X_train).to(device)
  y_train_tensor = torch.tensor(y_train).to(device)

  dataset = TensorDataset(X_train_tensor, y_train_tensor)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

  model.train()
  for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in dataloader:
      optimizer.zero_grad()
      outputs = model(batch_X)
      loss = criterion(outputs, batch_y)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    if (epoch+1) % 10 == 0:
      print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}")

  model.eval()
  return model

In [8]:
class LSTMClassifier:
  """ SkLearn-compatable wrapper forLSTM model """

  def __init__(self, seq_length=20, epochs=30, batch_size=32, lr=0.001):
    self.seq_length = seq_length
    self.epochs = epochs
    self.batch_size = batch_size
    self.lr = lr
    self.model = None
    self.scaler = MinMaxScaler()
    self.input_size = None
    self.classes_ = np.array([0, 1])

  def fit(self, X, y):
    """ fit the LSTM model """
    X_scaled = self.scaler.fit_transform(X)
    self.input_size = X_scaled.shape[1]
    X_seq, y_seq = create_sequences(X_scaled, y, seq_length=self.seq_length)

    if len(X_seq) == 0:
      self.model = None
      return self

    self.model = train_lstm_model(
        X_seq, y_seq,
        input_size=self.input_size,
        epochs=self.epochs,
        batch_size=self.batch_size,
        lr=self.lr
    )

    return self

  def predict_proba(self, X):
    """ predict probabilities """
    if self.model is None:
      return np.column_stack([np.zeros(len(X)) * 0.5, np.ones(len(X)) * 0.5])

    X_scaled = self.scaler.transform(X)

    # For prediction, we need to create a sequence ebding at each point
    # If we don't have enough history, pad with zeros
    if len(X_scaled) < self.seq_length:
      padding = np.zeros((self.seq_length - len(X_scaler), X_scaled.shape[1]))
      X_padded = np.vstack([padding, X_scaled])
    else:
      X_padded = X_scaled

    # create sequence for the last part
    X_seq = X_padded[-self.seq_length].reshape(1, self.seq_length, -1)

    # Predict
    self.model.eval()
    with torch.no_grad():
      X_tensor = torch.FloatTensor(X_seq).to(device)
      probs = self.model(X_tensor).cpu().numpy()[0, 0]

    # return probability for both classes
    return np.array([[1 - prob,prob]])

  def predict(self, X):
    """ predict classes """
    probs = self.predict_proba(X)
    return (proba[:,1] >   0.5).astype(int)

print("LSTM Classifier wrapper defined!")

LSTM Classifier wrapper defined!


**training models**

In [9]:
def train_models(feat_df):
  models = {}
  lstm_models = {}
  scores = {}

  tscv = TimeSeriesSplit(n_splits=N_SPLITS)
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if any(x in c for x in ['_freq', '_lag', '_gap', '_ema', '_streak', '_cooccur', 'prev_', '_decade_'])]
  print(f"Using {len(feature_cols)} features for training")
  print(f"Training unified models for all 7 numbers (6  main + 1 additional)...")

  for i in range(1, NUM_RANGE+1):
    target = f"num_{i}"
    x = feat_df[feature_cols]
    y = feat_df[target]

    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE
    )

    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        use_label_encoder=False,
        eval_metric='logloss',
        verbose=0
    )

    model = VotingClassifier(
        estimators=[
            ('rf', rf_model),
            ('gb', gb_model),
            ('xgb', xgb_model)
        ],
        voting='soft',
    )

    fold_scores = []
    for train_idx, val_idx in tscv.split(x):
      x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
      y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

      if len(y_train.unique()) < 2:
        continue

      try:
        model.fit(x_train, y_train)
        y_pred = model.predict_proba(x_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, y_pred))
      except Exception as e: continue

      # train final model on all data  AFTER CV loop
      # Check if we have both classes in the full dataset
      if len(y.unique()) < 2:
        from sklearn.dummy import DummyClassifier
        model = DummyClassifier(strategy='most_frequent')

      try:
        model.fit(x, y)
      except Exception as e:
        # fallback todummy classifierif  fitting fails
        from sklearn.dummy import DummyClassifier
        model = DummyClassifier(strategy='most_frequent')
        model.fit(x, y)

      models[i] = model
      scores[i] = np.mean(fold_scores) if fold_scores else 0.5

  if len(y.unique())>=2:
    try:
      lstm_clf = LSTMClassifier(seq_length=20, epochs=30, batch_size=32)
      lstm_clf.fit(x, y)
      lstm_models[i] = lstm_clf
    except Exception as e:
      lstm_models[i] = None
  else:
    lstm_models[i] = None

  if i % 10 == 0: print(f"Trained models from number 1-{i}...")
  avg_auc = np.mean(list(scores.values()))
  print(f"Average AUC (main numbers): {avg_auc:.4f}")
  print(f"Best performing numbers: {sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]}")

  return {'ensemble':models, 'lstm':lstm_models}


**predict probability**

In [10]:
def predict_probs(models, latest_row, full_df=None, feature_cols=None, lstm_weight=0.3):
  probs = {}
  ensemble_models = models['ensemble']
  lstm_models = models.get('lstm', {})

  for i in range(1, NUM_RANGE+1):
    # get ensemble prediction
    ensemble_model = ensemble_models.get(i)
    ensemble_prob = ensemble_model.predict_proba(latest_row)[0][1]

    # get LSTM prediction
    lstm_model = lstm_models.get(i)

    if lstm_model is not None and full_df is not None and feature_cols is not None:
      try:
        # use last seq_length rows for LSTM prediction
        seq_length = lstm_model.seq_length
        if len(full_df) >= seq_length:
          lstm_input = full_df[feature_cols].iloc[-seq_length:]
          lstm_prob = lstm_model.predict_proba(lstm_input)[0][1]

          # weighted average of ensemble and LSTM
          probs[i]  = (1 - lstm_weight) * ensemble_prob + lstm_weight * lstm_prob
        else:
          probs[i] = ensemble_prob
      except Exception as e:
        probs[i] = ensemble_prob
    else:
      probs[i] = ensemble_prob

  return probs

def predict_probs_ensemble(models, latest_row):
  probs = {}
  ensemble_models = models['ensemble'] if isinstance(models, dict) and 'ensemble' in models else models
  for i in range(1, NUM_RANGE+1):
    ensemble_model = ensemble_models.get(i)
    ensemble_prob = ensemble_model.predict_proba(latest_row)[0][1]
    probs[i] = ensemble_prob
  return probs

**monte carlo**

In [11]:
def monte_carlo_population(probs, size=1000):
  numbers = list(probs.keys())
  weights = np.array(list(probs.values()))
  weights = weights / weights.sum()
  population = set()

  while len(population) < size:
    combo = tuple(sorted(np.random.choice(numbers, 6, replace=False, p=weights)))
    population.add(combo)

  return [list(c) for c in population]

def compute_mc_frequency(population):
  counter = Counter(tuple(sorted(p)) for p in population)
  total = sum(counter.values())
  return {k:v/total for k, v in counter.items()}

**ga component**

In [12]:
def fitness(combo,  probs, mc_freq):
  combo_tuple = tuple(combo)
  prob_score = sum(probs[n]  for n  in  combo)
  mc_score = mc_freq.get(combo_tuple, 0)
  consecutive_penalty = sum(1 for i in range(len(combo)-1) if combo[i]+1 == combo[i+1]) * 0.02

  # Sum range penalty
  s = sum(combo)
  if s < 90 or s > 230:
    sum_penalty = 0.1
  elif s < 115 or s > 200:
    sum_penalty = 0.05
  else:
    sum_penalty = 0

  # Odd/Even balance
  odd_count = sum(n % 2 for n in combo)
  if odd_count < 2 or odd_count > 5:
    balance_penalty = 0.05
  else:
    balance_penalty = 0

  # High/low balance
  low_count = sum(1 for n in combo if n <= 25)
  if low_count < 2 or low_count > 5:
    range_penalty = 0.03
  else:
    range_penalty = 0

  # decades spread bonus
  decades_covered = len(set((n-1)//10 for n in combo))
  decade_bonus = 0.02 * (decades_covered - 3) if decades_covered >= 3 else -0.02

  # prime number bonus - historically primes appear slightly more often
  primes = {2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47}
  prime_count = sum(1 for n in combo if n in primes)
  prime_bonus = 0.01 * (prime_count - 2) if 1<=prime_count <= 5 else -0.01

  # Number spacing bonus - reward evenly spaced number
  sorted_combo = sorted(combo)
  gaps = [sorted_combo[i+1] - sorted_combo[i] for i in range(len(sorted_combo)-1)]
  avg_gap = sum(gaps)/ len(gaps)
  gap_variance = sum((g-avg_gap)**2 for g in gaps) / len(gaps)
  spacing_bonus = 0.02 if 4 <= avg_gap <= 10 and gap_variance < 30 else 0

  # final score: heavily weight the probability score
  return prob_score * 0.6 + mc_score * 0.2 + decade_bonus + prime_bonus + spacing_bonus - consecutive_penalty - sum_penalty - balance_penalty - range_penalty

def tournament_selection(population, scores, k=3):
  selected = random.sample(list(zip(population, scores)), k)
  selected.sort(key=lambda x: x[1], reverse=True)
  return selected[0][0]

def crossover(p1, p2):
  child = list(set(p1[:4] + p2[4:]))
  while len(child) < 7:
    n = random.randint(1, 49)
    if n not in child:
      child.append(n)
  return sorted(child[:7])

def mutate(ind, rate=0.2):
  if random.random() < rate:
    idx = random.randint(0, 6)
    new_val = random.randint(1, 49)
    while new_val in ind:
      new_val = random.randint(1, 49)
    ind[idx] = new_val
  return sorted(ind)


**hybrid ga**

In [13]:
def run_hybrid_ga(probs, generation=50, pop_size=300):
  population = monte_carlo_population(probs, size=pop_size)
  mc_freq = compute_mc_frequency(population)
  for gen in range(generation):
    scores = [fitness(ind, probs, mc_freq) for ind in population]
    ranked = sorted(zip(population, scores), key=lambda x: x[1], reverse=True)
    elites = [r[0] for r in ranked[:20]]
    new_population = elites.copy()
    while len(new_population) < pop_size:
      p1 = tournament_selection(population, scores)
      p2 = tournament_selection(population, scores)
      child = crossover(p1, p2)
      child = mutate(child)
      new_population.append(child)
    population = new_population
    if gen % 10 == 0:
      print(f"Gen {gen} Best Score: {ranked[0][1]:.4f}")
  final = [(ind, fitness(ind, probs, mc_freq)) for ind in population]
  final.sort(key=lambda x: x[1], reverse=True)
  return final[:10]


**backtest (real validation)**

In [14]:
def backtest(models, feat_df, verbose=True, use_lstm=True):
  """
  Backtest unified 7-number predictions with ensemble + LSTM

  Args:
     models: Dictionary with 'ensemble' and 'lstm' model ditionalries
     feat_df: Feature dataframe
     verbose: Print progress updates
     use_lstm: Whether to use LSTM prediction in combination with ensemble
  """
  hits = []
  hit_distribution = {0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 6:0, 7:0}
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if any(x in c for x in ['_freq', '_lag', '_gap', '_ema', '_streak', '_cooccur', 'prev_', '_decade_'])]
  start_idx = 50
  for i in range(start_idx, len(feat_df)-1):
    X = feat_df.iloc[[i]][feature_cols]
    actual = feat_df.iloc[i+1]
    if use_lstm:
      probs = predict_probs(models, X, feat_df.iloc[:i+1], feature_cols, lstm_weight=0.3)
    else:
      probs = predict_probs_ensemble(models, X)
    pred = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:7]
    pred_nums = set(n for n,_ in pred)
    actual_nums = set(j for j in range(1, 50) if actual[f'num_{j}'] == 1)
    hit_count = len(pred_nums & actual_nums)
    hits.append(hit_count)
    hit_distribution[min(hit_count, 7)] += 1

    if verbose and (i - start_idx + 1) % 100 == 0:
      print(f"Progress {i - start_idx + 1}/{len(feat_df) - start_idx -1}, Avg hits:{np.mean(hits):.4f}")

    print(f"Total prediction: {len(hits)}")
    print(f"average hit per draw: {np.mean(hits):.4f}")
    print(f"Expected by random chances: {7*7/49:.4f}")
    print(f"hit distribution: {hit_distribution}")

    return np.mean(hits)

**main**

In [ ]:
def main():
  df = load_data('/content/drive/MyDrive/Colab Notebooks/sg_pools/ToTo.csv')
  bin_df = create_binary_matrix(df)
  feat_df = add_features(bin_df)
  models = train_models(feat_df)

  # backtest
  backtest(models, feat_df)

  # latest prediction
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if any(x in c for x in ['_freq', '_lag', '_gap', '_ema', '_streak', '_cooccur', 'prev_', '_decade_'])]
  latest = feat_df.iloc[[-1]][feature_cols]

  probs = predict_probs(models, latest, feat_df, feature_cols, lstm_weight=0.3 )

  # display individual number probability
  sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
  for rank, (num, prob) in enumerate(sorted_probs[:15], 1):
    bar = "❚"*int(prob*100)
    print(f"#{rank:.2d} Number {num:.2d} :{prob:.4f} {bar}")

  # hybrid optimization
  results = run_hybrid_ga(probs)

  print("\nTop combinations:")
  for rank, (combo, score) in enumerate(results, 1):
    combo_probs = [probs[n]  for n in combo]
    avg_prob = sum(combo_probs) / len(combo_probs)
    print(f" #{rank} : {combo} | Fitness: {score:.4f} | Avg Prob: {avg_prob:.4f}")

  if results:
    best = results[0][0]
    print(f"Predicted 7 numbers : {best}")
    print(f" Good Luck! 🍀")

  return models, probs, feat_df


if __name__ == "__main__":
  models, probs, feat_df = main()

Using 23373 features for training
Training unified models for all 7 numbers (6  main + 1 additional)...
